# Target Selection task API examples

This notebook demonstrates the independent task-level API. Each section can be run separately: target collection, source merging, local visibility, Rubin coverage/photometry, and reports. The cells that access MOP, TAP, or Butler are intentionally not executed automatically because they require an RSP session and can be expensive.

## 1. Imports and run settings

Change these variables directly in the notebook. No monolithic `run_analysis` call is required.

In [1]:
from pathlib import Path

import pandas as pd

from target_selection import (
    collect_mop_targets,
    load_target_list,
    merge_targets,
    restrict_targets,
    evaluate_visibility,
    save_visibility_plots,
    make_visibility_sequence,
    query_lsst_coverage,
    compute_lsst_photometry,
    save_lsst_photometry,
    create_target_report,
    create_target_reports,
    create_lightcurve_plots,
    create_lightcurves_report,
    plot_sky_dual_metric,
)
from target_selection.run_paths import create_run_structure
from target_selection.sources.lsst import create_tap_service, create_butler

START_DATE = "2026-08-17"
END_DATE = START_DATE
OBSERVATORY = "El Leoncito"
OUTPUT_DIR = Path("outputs/task_api_example")
DATA_RELEASE = "DP2"
MAX_WORKERS = 4

paths = create_run_structure(OUTPUT_DIR, START_DATE, END_DATE, data_release=DATA_RELEASE)
paths

{'run': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00'),
 'tables': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00/tables'),
 'sky_plots': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00/sky_plots'),
 'visibility_plots': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00/visibility_plots'),
 'lightcurves': PosixPath('outputs/task_api_example/lightcurves'),
 'monitoring_reports': PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00'),
 'target_reports': PosixPath('outputs/task_api_example/target_reports'),
 'targets': PosixPath('outputs/task_api_example/target_reports'),
 'targets_visibility_selected': PosixPath('outputs/task_api_example/target_reports')}

## 2. Collect targets from providers

MOP is optional. A local CSV can be loaded with a column map when its names are not already normalized. Follow-up inventories such as HSH can be imported through the configured source adapters or the registry workflow.

In [2]:
# Requires an initialized MOP client or an active RSP/MOP environment.
mop_targets = collect_mop_targets(
    start_date=START_DATE, end_date=END_DATE, observatory=OBSERVATORY,
    cache_dir=OUTPUT_DIR, max_workers=MAX_WORKERS,
)

# Example local provider list. Required normalized fields: Target, RA_deg, Dec_deg.
csv_targets = load_target_list(
    "/home/karennowo/target_selection/hsh_data/hsh_objects.csv",
    source_name="hsh",
    column_map={"Target": "object", "RA_deg": "ra_deg", "Dec_deg": "dec_deg"}
)

# Combine any number of provider/survey target tables. Higher coordinate priority wins.
targets = merge_targets(mop_targets, csv_targets)
print(f"MOP targets returned: {len(mop_targets)}")
print(f"Local target-list rows: {len(csv_targets)}")
print(f"Merged targets: {len(targets)}")
targets = restrict_targets(targets, target_names=["OGLE-2025-BLG-0451"])
print(f"Targets after restriction: {len(targets)}")
if not targets.empty:
    print(targets[["Target", "RA_deg", "Dec_deg"]].to_string(index=False))


[targets] Querying MOP visibility for 2026-08-17 to 2026-08-17.
[targets] MOP targets with event data: 50
[targets] Loading hsh targets from /home/karennowo/target_selection/hsh_data/hsh_objects.csv.
[targets] Normalized targets: 49
[targets] Merged unique targets: 97
MOP targets returned: 50
Local target-list rows: 49
Merged targets: 97
[targets] Restriction kept 1/97 targets.
Targets after restriction: 1
            Target     RA_deg  Dec_deg
OGLE-2025-BLG-0451 269.540958  -19.978


## 3. Evaluate local visibility

The visibility task applies astronomical-night and observing-window constraints, then the altitude and minimum-duration selection criteria.

In [3]:
visibility = evaluate_visibility(
    targets, start_date=START_DATE, end_date=END_DATE,
    observatory=OBSERVATORY, minimum_altitude_deg=40.0,
    minimum_observable_minutes=90.0, time_step_minutes=1,
    observing_windows=("20:00", "07:00"),
    output_path=paths["visibility_plots"] / "visibility_selection.csv",
)
visibility.head()

# Save nightly plots, optionally limiting each panel to a maximum number of targets.
nightly_plots = save_visibility_plots(
    targets, start_date=START_DATE, end_date=END_DATE,
    output_dir=paths["visibility_plots"], observatory=OBSERVATORY,
    minimum_altitude_deg=40.0, minimum_observable_minutes=90.0,
    observing_windows=("20:00", "07:00"), max_targets_per_plot=20,
    selection=visibility,
)

# Stack the already computed visibility table into a PDF or PNG sequence.
sequence = make_visibility_sequence(
    visibility, paths["visibility_plots"] / "visibility_sequence.pdf",
    observatory=OBSERVATORY, observing_windows=("20:00", "07:00"),
    start_date=START_DATE, end_date=END_DATE,
)


[visibility] Evaluating 1 targets from 2026-08-17 to 2026-08-17.
[visibility] Rows: 0; selected: 0.
      Visibility selection was empty; recomputing it from the supplied targets.
      visibility: 1/1 (2026-08-17, 0/0 selected, 1 plot(s))


## 4. Add Rubin/LSST context

TAP returns visit/detector coverage. Butler is used for image/coadd products and the configured photometry method. These are optional tasks, so the same target table can be analyzed without a reference collection.

In [4]:
tap_service = create_tap_service(DATA_RELEASE)
coverage = query_lsst_coverage(
    targets, tap_service=tap_service, data_release=DATA_RELEASE,
    max_workers=MAX_WORKERS,
)
coverage.to_csv(paths["tables"] / "release_coverage.csv", index=False)

# Select one method in the call: dia_forced_catalog, coadd_forced, or calexp_forced.
release_photometry = compute_lsst_photometry(
    targets, coverage=coverage, method="dia_forced_catalog",
    tap_service=tap_service, data_release=DATA_RELEASE,
    max_workers=MAX_WORKERS,
)
save_lsst_photometry(release_photometry, paths["tables"] / "release_photometry.csv")


[lsst] Querying DP2 coverage for 1 targets.
[lsst] Coverage rows: 92 (1 targets).
[lsst] Photometry rows returned: 24 using dia_forced_catalog.
[lsst] Saved 24 photometry rows to outputs/task_api_example/2026-08-17__obs_18-00_to_06-00/tables/release_photometry.csv.


PosixPath('outputs/task_api_example/2026-08-17__obs_18-00_to_06-00/tables/release_photometry.csv')

## 5b. HSH targets: MOP + HSH epochs + DP2 DIA curves

This optional batch task uses every target present in the HSH inventory. It queries DP2 coverage and retrieves both published DIA products: `ForcedSourceOnDiaObject` and `DiaSource`. The two measurements are plotted with different markers. HSH epochs are read from the local registry, while MOP photometry is loaded from the persistent MOP cache when available.


In [5]:
# This cell is intentionally not executed automatically: it can issue many TAP queries.
from target_registry import TargetRegistry
from target_selection.tasks.surveys import import_hsh_inventory, observed_targets

# Resolve repository paths whether Jupyter uses the repository root or notebooks/ as cwd.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "hsh_data").exists() and (PROJECT_ROOT.parent / "hsh_data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
HSH_INVENTORY = PROJECT_ROOT / "hsh_data" / "image_collection_astro.csv"
RUN_OUTPUT_DIR = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else PROJECT_ROOT / OUTPUT_DIR
HSH_DATABASE = RUN_OUTPUT_DIR / "hsh_target_database.sqlite"
HSH_LIGHTCURVE_DIR = RUN_OUTPUT_DIR / "hsh_lightcurves"

# Import/update local HSH observation epochs, then obtain all observed HSH targets.
import_hsh_inventory(HSH_INVENTORY, database_path=HSH_DATABASE, force=False)
hsh_targets = observed_targets(providers=("HSH",), database_path=HSH_DATABASE)
# The HSH inventory identifies observed objects but its image CRVAL values are
# pointings, not event coordinates. Complete positions from the curated HSH list.
hsh_coordinate_targets = load_target_list(
    PROJECT_ROOT / "hsh_data" / "hsh_objects.csv",
    source_name="HSH",
    column_map={"Target": "object", "RA_deg": "ra_deg", "Dec_deg": "dec_deg"},
)
hsh_targets = hsh_targets.drop(columns=["RA_deg", "Dec_deg"], errors="ignore").merge(
    hsh_coordinate_targets[["Target", "RA_deg", "Dec_deg"]],
    on="Target", how="left", validate="one_to_one",
)
missing_hsh_coordinates = hsh_targets["RA_deg"].isna() | hsh_targets["Dec_deg"].isna()
print(f"HSH targets selected: {len(hsh_targets)}; missing coordinates: {int(missing_hsh_coordinates.sum())}")

# Query DP2 visit coverage once for all HSH targets.
hsh_coverage = query_lsst_coverage(
    hsh_targets, data_release=DATA_RELEASE, max_workers=MAX_WORKERS,
    output_path=RUN_OUTPUT_DIR / "hsh_dp2_coverage.csv",
)

# Retrieve both DiaSource and ForcedSourceOnDiaObject products.
hsh_dia_photometry = compute_lsst_photometry(
    hsh_targets, coverage=hsh_coverage, data_release=DATA_RELEASE,
    method="dia_combined_catalog", max_workers=MAX_WORKERS,
    output_path=RUN_OUTPUT_DIR / "hsh_dp2_dia_photometry.csv",
    persistent_target_dir=RUN_OUTPUT_DIR / "rubin_photometry",
)

# Load HSH epochs from the same local registry.
hsh_epochs = TargetRegistry(HSH_DATABASE).observation_epochs(hsh_targets, providers=("HSH",))

# Persistent PNGs: MOP points (if cached), HSH coverage, and both DP2 curves.
hsh_lightcurve_pngs = create_lightcurve_plots(
    hsh_targets, output_dir=HSH_LIGHTCURVE_DIR,
    mop_photometry_dir=PROJECT_ROOT / "outputs" / "mop_photometry",
    release_photometry=hsh_dia_photometry, coverage=hsh_coverage,
    observatory_epochs=hsh_epochs, data_release=DATA_RELEASE,
    layers=("mop_photometry", "release_photometry", "release_epochs", "hsh"),
    overwrite=False, verbose=True,
)
print(f"HSH light-curve plots: {len(hsh_lightcurve_pngs)}")


[surveys] Importing HSH inventory: /home/karennowo/target_selection/hsh_data/image_collection_astro.csv
[surveys] HSH records imported/updated: 0
[surveys] Previously observed targets: 29
[targets] Loading HSH targets from /home/karennowo/target_selection/hsh_data/hsh_objects.csv.
[targets] Normalized targets: 49
HSH targets selected: 29; missing coordinates: 0
[lsst] Querying DP2 coverage for 29 targets.
[lsst] Coverage rows: 2586 (24 targets).
[lsst] DiaSource rows returned: 159 (142 measured; 29 targets).
[lsst] Photometry rows returned: 989 using dia_combined_catalog.
[lsst] Saved 989 photometry rows to /home/karennowo/target_selection/outputs/task_api_example/hsh_dp2_dia_photometry.csv.


/home/karennowo/target_selection/target_selection/sources/lsst.py:185: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty, ignore_index=True, sort=False) if nonempty else pd.DataFrame()


[reports] Updating 29 light-curve PNG(s).


Light curves:   0%|          | 0/29 [00:00<?, ?target/s]

[reports] Light-curve PNGs ready: 29 in /home/karennowo/target_selection/outputs/task_api_example/hsh_lightcurves
HSH light-curve plots: 29


## 5c. HSH target reports without visibility selection

This task is independent of visibility. It creates one dashboard per HSH-observed target, using the DP2 coverage and DIA photometry already computed above. Targets without a covering DP2 coadd are reported and skipped.


In [8]:
# Uses hsh_targets, hsh_coverage, and hsh_dia_photometry from section 5b.
hsh_target_reports = create_target_reports(
    hsh_targets,
    coverage=hsh_coverage,
    data_release=DATA_RELEASE,
    release_photometry=hsh_dia_photometry,
    output_dir=RUN_OUTPUT_DIR / "target_reports",
    mop_photometry_dir=PROJECT_ROOT / "outputs" / "mop_photometry",
    max_workers=MAX_WORKERS,
    overwrite=True,
    verbose=True,
)
print(f"HSH target reports generated or reused: {len(hsh_target_reports)}")


[reports] Creating 29 target report(s).
[reports] Target reports are generated sequentially; max_workers is ignored.


Target reports:   0%|          | 0/29 [00:00<?, ?target/s]

[reports] Building target report: ASASSN-24fs
[reports] Report saved: /home/karennowo/target_selection/outputs/task_api_example/target_reports/ASASSN-24fs_target_report.png
[reports] Building target report: Gaia23cnu
Photometry skipped for Gaia23cnu: no MOP client or cached photometry
[reports] Report saved: /home/karennowo/target_selection/outputs/task_api_example/target_reports/Gaia23cnu_target_report.png
[reports] Building target report: Gaia23dpn
Photometry skipped for Gaia23dpn: no MOP client or cached photometry
[reports] Skipping Gaia23dpn: No coadd covers target 'Gaia23dpn'
[reports] Building target report: Gaia24bsi
[reports] Skipping Gaia24bsi: No coadd covers target 'Gaia24bsi'
[reports] Building target report: Gaia24bsj
[reports] Skipping Gaia24bsj: No coadd covers target 'Gaia24bsj'
[reports] Building target report: Gaia24ddl
[reports] Skipping Gaia24ddl: No coadd covers target 'Gaia24ddl'
[reports] Building target report: OGLE-2023-BLG-0723
[reports] Report saved: /home/k

In [9]:
  hsh_dia_photometry = compute_lsst_photometry(
      hsh_targets,
      coverage=hsh_coverage,
      data_release=DATA_RELEASE,
      method="dia_combined_catalog",
      max_workers=MAX_WORKERS,
      output_path=RUN_OUTPUT_DIR / "hsh_dp2_dia_photometry.csv",
      persistent_target_dir=RUN_OUTPUT_DIR / "rubin_photometry",
  )

  hsh_lightcurve_pngs = create_lightcurve_plots(
      hsh_targets,
      output_dir=HSH_LIGHTCURVE_DIR,
      mop_photometry_dir=PROJECT_ROOT / "outputs" / "mop_photometry",
      release_photometry=hsh_dia_photometry,
      coverage=hsh_coverage,
      observatory_epochs=hsh_epochs,
      data_release=DATA_RELEASE,
      layers=(
          "mop_photometry",
          "release_photometry",
          "release_epochs",
          "hsh",
      ),
      overwrite=True,
      verbose=True,
  )

[lsst] DiaSource rows returned: 159 (142 measured; 29 targets).
[lsst] Photometry rows returned: 989 using dia_combined_catalog.
[lsst] Saved 989 photometry rows to /home/karennowo/target_selection/outputs/task_api_example/hsh_dp2_dia_photometry.csv.


/home/karennowo/target_selection/target_selection/sources/lsst.py:185: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty, ignore_index=True, sort=False) if nonempty else pd.DataFrame()


[reports] Updating 29 light-curve PNG(s).


Light curves:   0%|          | 0/29 [00:00<?, ?target/s]

Photometry skipped for Gaia23cnu: no MOP client or cached photometry
Photometry skipped for Gaia23dpn: no MOP client or cached photometry
[reports] Light-curve PNGs ready: 29 in /home/karennowo/target_selection/outputs/task_api_example/hsh_lightcurves


## 5. Generate products

Reports can be generated for all targets or for one selected target. Individual target reports are stored in the shared `target_reports` folder.

In [ ]:
target_reports = create_target_reports(
    targets, coverage=coverage, data_release=DATA_RELEASE,
    release_photometry=release_photometry, output_dir=paths["target_reports"],
    max_workers=MAX_WORKERS, overwrite=False,
)

# One target only:
one_target = targets.loc[targets["Target"].eq("OGLE-2025-BLG-0451")].iloc[0]
report = create_target_report(
    one_target, coverage=coverage, data_release=DATA_RELEASE,
    release_photometry=release_photometry, output_dir=paths["target_reports"],
)

# A sky map is a product task too; metrics are columns in the merged target table.
plot_sky_dual_metric(
    targets, "mag_now", "release_n_visits",
    paths["sky_plots"] / "sky_by_mag_and_visits.png",
    title=f"{DATA_RELEASE} target coverage", marker_encoding="split_color",
)

# Light curves can be generated from the normalized provider/reference tables.\n",

lightcurve_pngs = create_lightcurve_plots(
    targets, output_dir=paths["lightcurves"], mop=None,
    release_photometry=release_photometry, coverage=coverage,
    data_release=DATA_RELEASE, overwrite=False,
)

selected_targets = visibility.loc[visibility["selected_for_visibility"].astype(bool)].drop_duplicates("Target")
lightcurves = create_lightcurves_report(
    selected_targets, output_path=paths["run"] / "lightcurves.pdf",
    data_release=DATA_RELEASE, release_photometry=release_photometry,
)


## 6. Inspect the run

The run directory contains only the products requested by the cells. The shared target database/cache can be reused by later runs.

In [ ]:
for path in sorted(paths["run"].rglob("*")):
    if path.is_file():
        print(path)
